<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/MiniMax_H3_turbo4%2B%E5%AF%BC%E6%BC%94%E5%8F%B0%E7%BB%BC%E5%90%88%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax H3 Turbo + 导演台（Colab A100）
按 Cell 1→4 运行。工作流来自随附 JSON；旧素材引用已清空，首尾帧请直接在 ComfyUI 的 `MiniMaxH3Director` 时间线上传。ComfyUI、Director、KJNodes、Turbo 节点和 Manager 每次都会同步最新提交。

In [ ]:
# Cell 1：A100 检查与配置
import os,sys,re,json,gzip,base64,shutil,subprocess,time,urllib.request
from pathlib import Path
ROOT=Path('/content'); COMFY=ROOT/'ComfyUI'; PORT=8188
WORKFLOW='MiniMaxH3_导演台全能工作流_A100'
HF=ROOT/'hf_cache'; HF.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(HF); os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
def sh(cmd,cwd=None,check=True):
 print('+',cmd); return subprocess.run(cmd,shell=True,cwd=str(cwd) if cwd else None,text=True,check=check,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
q=sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits').stdout.strip().split(',')
name=q[0].strip(); vram=float(q[1])/1024
print(f'GPU={name}, VRAM={vram:.1f}GB, free disk={shutil.disk_usage(ROOT).free/1024**3:.1f}GB')
if 'A100' not in name.upper() or vram<38: raise RuntimeError('请选择 Google Colab A100 40GB + 高 RAM 运行时')
try:
 from google.colab import userdata
 token=userdata.get('HF_TOKEN')
 if token: os.environ['HF_TOKEN']=token
except Exception: pass
PAYLOAD='H4sIAACRg2oC/+1ce3MUR5L/KhNzERfei9HQ74cidi9kELbuEPKKx92GRXS0ZnqkXkYz45keSRgTIWB5GvPYxcb2gbENBmzOIL9Ay8OOuI+yVs+M/vJX2MyqflT3lKb1wHve23MYmM6szqzK+lVVZlVlH81X7ZZnVd3aYcst54clrZCv1ctOKz/8+tH8HPzKDwuFfL3tNdoeJdbsOSDmd+4Zey1fyOObSBe1Q/BgTzvVmOcdabAl6yW76r7plC1WwjF4rQVUECFrQkEUUE6jjiKHdEkoSJIAhAW3PON4LWverrZJ3fJvLDg1eb5qydK0NefW3Dl70ZqVLbfmGVapXptv1r1iy644nlNr1Zst0B+Ugl9lp2K3q14eBLu1qF1UR344amGp6jZoXY8V+mlxa1la2OSJ8ZcneG1mZXJUktdjbeQxVhQ8ZukIhHDEl515t0QUtGZtFKIXUqxYWUTIUhcKBWtWqvYMGPMoKCBgSkBgT90uO00U6HrVkJh7KeiW3I7cb7FLhw7u+RWUaTTrDafpudjZFIfVBPo23/1lt+mUvHrzCFrIWfQsp1YCscA7FgHMajv4dg0K2tNYx6NMhw17zbYTtoj+DppOnqDNe0FeDsvmKvVmbt8/TyZbDiXqTfwxLGDvDBpcB0dG2bGlMmOLsgK7BuXSXYLk1MBSjeS4kg3+uGKMOe+WnTqwHKvSELWEOTNGDr6TGjgRKW4JQ8rCWCywH2Uya40+kB3ERuSonbJAtbG2J6EERQYDKKr6IJTENY9BIm4SJPKLB4kqZILEbpfd0FCy9IsFiTIQJCPYiM2DZEDb/1YgkbJAsm//5NjeVxiDRoTAHnGBtD0DDoMRTYEFmWCCgsQUtKIsi4KqaIauFkyxqMuCIZq6IImKoSg88CRREahq1dtNuvoAmqFVQlzfiBXU9194VQ0KcTreiF58rQmztbMwUjvC9PyuoJdyvR8u9z650Dn/mX/lfB8K1uu3dTqK0cQM58yuOji2a3SCGdEmO6ADXgjisGjfoCYMpsskA5wqKdllhqibpixCb2qSWpAVuahpiqgqgirpMr/XJKVgcHsOxsKM04p6TtTjOkesoNJj4yOvcGeioCDap88vIWMsFm/E4kNOIH3kwK4xrkloOa4/VGm0mNkHnyJFZqyI0gM1u/dMjOznqaGyOEqmXc8qOw1vlut3xdxYIUsLbbeXq5SR3Q99LZ4ym47tOWQp3DC2SYMCPydWM9DVYbQwuCfrWAx3qGepXq0DJ/9PkiTleevLW7k5Z8ZuuIsw9+beyo20GlAv+DFBpIDPCN6726g6v5alX+Xemqq9NTQ0FP2Bx5xQlKC4qA2b8I8mGLnFnKxKuYAnxzxd1oCniFrIU2KeoSnIM4SQp8Y8UxOApypKyNNiHsyHKBTVBkydYYpQD2AqkVSDYUoieVOPqmqyTAN1Yo1DJvsqDFzkaoFSsSiwTF1LMhnziKqAbxqyFDKZdooaMYIpRW+yOg0JmaIgBWaQEkpNSSBcg7ybcAeSYA1QOm43D5frC7W9dc9hpmh8HM7tA9jn9jme59ZmWrlJp+I0wYHHctGUOj0TAUsQhHzs2QjgyYhS7NpoklyUFckwVVXQDRHmQFGQgSRJsqFrpioD6RAdQ6K84bUghrycNdNT77LdrLKODkPLXJrjwqiKJaYlbmDNoOXYNUMUCoqMS7sMi4MqqIZqKpH5AKQAWlGSdd2UTLLwy4pRlA1RNFVBM2RauH9UEz07xsF/GrcXX5WtcO21PGke6mW3vXr4D3epidpHZmhzyw3lrgRu1cGfVqPpVNxFdlVIcZh1oY+T2W/9anh1qTfnbI+tAiUwmkNClkscieJtbwBAS+zWBnlmtjWC51jH7t9Zu363d2R8bCdRaB2UuVscVG7/eqTHBrLnN7saUanBepQ2Y0imrR20RsWamRUqc7yOT+wa3cN4ZpLIuGYhM5zFwrJpu1BGKuIypGTElR2VV6rSvA0Nb9dA9KD9jo1tcoEYLx2GxbS4lSwtC3WsTI7KBcedmQWHIrXZlSDHilPkLN1J4f0gFCMRB/aO7u+LBIPZKfeqnEP+5uLBDXcNExqW3Uql3XLrNSsQPTBOTLQuQH1s7UHAZ1obI1+Jga9sDPjSiwC+pCeBL0tmUdVFRQf3R6crypCoqUXD1CBKUSVwTQSZOy7KbgtNU84XKna15XCXDWLXOLTU4gaEnKwG0HJcLLcgcrFsD/rWgz5k0JxiMPFsmpGF6D4VnGrY1Wp9AcAy14CZkRtnJEswUVSKHtTm5YmJPaMje7nxVFIXZ4jFE/1rtjdbmt0HLRgJG/Dv/7bt8JorNEa1noVqFhEE1ezW2SZBwUznql4URcUAX1vRcPejIDFnGQq46xJi2ZQFDSNvgLgqF3VBMRVBN1XD3Mx+SQrT0naqn+69OGyMXLVxZw5mq9FKxS25YPCE6aE3SrPb7dJNaIr72Vw/sFRkfmAZ6fEfPus8u+pfWu5c+uPq00f+9WX/xtJPzz6c9bxGa3jHjhnXm21PFwHlO0bGxiEQbe7YWZ+rHDkwZvU7sFO1qZp/50Tno+urz3/oXr3XlOanpubxryb8/dOzC2vXl3p3jvvLp/0n7/ZOX++882nn7GVwXWC5SL/r4Wsu/oWryXrvkpUGX+3dPb326ZXOux90773NVH9hYaE47VZd/EMaQX3vlw+K+99ol7SZ0cmXd+Dr3a/Pra7c6N655J/9HF7vnvizf+bpVN6/9M3qyvne3T/2znzhP/6q++Ef/Nvf+icvTeV/s3Ziqffwcffj47/xP7nvn/4geobXf1y6DAhbWKxLb/y4dGVt6bgIcdjkq/7KcdTVWfl87c69zqXL3dtPoLD/Xx9Pvtq9+ND/9KR/+uvuF8fXrj1Y+/R9aHHnwxOda4/AFqsr76w+Od158N3atW/9Z+9Cn/nnb/ZOPocy/qlH/oOT9NE/dW/t5PPOl7f8lRVgdW8sdZY+8y89XH36mX/9Xuf6f2PvPjjXu3Uq1euo6+x71O6rT6507z9ce/c6NDPHmnHRtWsz9drMkXaNWJJsVu0oO57tVndoWtmYnhacIcGcLg0pqqQOmbKtDhm6DNGvXAau8a/NX+tjv315fGzLETBruXUCXk2V44BX0TG+VQRVFXWVBriyoBRFQRFVDeYFf1CWYvURd1yHeMwqaJhUlTVJ1WVIkEZbjMBhWthAMq+s618EapRW4e4kk1NQPvZDtRI27nUg0GIe2s6PYt3NIZLIbt5vbPAwfmxjclOrtWiLqS1AHbwyyZbmGoDvmVmzyxHa7tWFjJ8szbWg6jXrTY1dZgTFLxM0MmYOS3EZQrRgFWhgI1ljHhsfbGHLYd5j1HSYzpSBqBnNGJulq0dQ1Ldo6L4iiVhQlSZRFWdE1VSMbSf3rEM7fub8sXc2t3XnPX/7eX7nbvXqTzuUv7XabLW9oj93ycruxF/E4HP5fO3Om8/Hj3oPvu88f5AsiDFINA+FFdH4lpaDqWkGX6R9ZB4fg6FR+3mliZDEFg7AwlXfKrjcO4xAep/ItZ2YOVtWpPDC8umdXiaoW8PDlKYqfSdvD0iB9iu6dwAOIxeh7L7CJICKBMHcDOSa12tOVehVmgJiE1idPZBIkpEqoFebD4GncboTPFFukZjsRyaQuBjDAdYFwqryP8Cdhfg5kHAvrsrPqNiKxM9X6tF2ldffs1uH9YT021QukvjD7zTW8uEmAkkhNM9yTPBibimOYfutx7JK2HjYMZljPrbXr7Va0+wlcEvdgI53aGKKX6iVAZvQei6z5H27ZmwWqrhkR7VUSVQIRd3JpS8iJKNs00qiIAAvhXL02WiPBF1MLSt9Zr1btRivmoHo689PqzYUwJPglLbbJHj8gzq0Tjjys5F7q3n+ve+5s5+Mz/pnTtAPicwEoJRSxtuFhAGIXoVuFxXq0TCyBg2EKhx9pMg6Rqfxs2FjKhNB9dBFnmGgACDhWCCkaLRDy0EqiVSIqmNxpkiES947rHeFZJeRNwJAE00S6VLRMs13bB4Auef1vRiw6jOmooEOX9AV2dZlUZq5V9hblN11NOuzNUwx5dtML2lN1ajPEBiLp4Ao7pCip3G6i7UFdKZgvGKyvriytrnzRvfCt/+VldKk+Or62dNP/7Grv8RXwoeBv8BB7P5wBH3R15cverdtQrHPxHJQEzwscSSiz+vz06tP3Vp++0/3Do+7Tu92nX9LXf1w6Eb24+uzbtQ8vgcuJZUDg4yudLz/zby2T1tSg4z133nktrtW0Xc4F01LCyLuhCB7BMnZ0W/vQGsTuU8HGCVJHa+UUbfA4YtBEB9AsO3SwM51aeWPvq6LEvq8ZEr6emJ+SUwysgMneNvTa7yuyPJPobdqb2+zvqKf9H+6zPe1futA7c4L2UOfKaf/8Pdr9/tllChH/68+7N7/pXDuOrvWpT+F/jB1uXF5deRtDmQ8voXN+7RH480DvfAcwuoElv/7AX36flu88Ouvf/gqJzy75Z570/nTDv/157+SFzgcPM7Hy49JxqgtU+EsfrD7B+RwrcGt57eY324BRhJcXjiI+CjYMonVAuAkULSweabdK9vzvEyiiy+wLQdHq04sUFRhRfve49+Q+Yob88C9f7H7+A9I/udW5/g7ST16Ax9U/3/EvnEK8XXwKoa3/4DaWefh9BKred8sAOcpaO3MBX0wCA8SCTMADCiTS8Pd3y73HDxEtBE6/VEhsd2LZAiZwWfHcOQdcdSda3tA7SmKVt36CkHF7EU9nIxK0lFYyOILYncIN1uawcyT2/dZZxawWUb+5pq638GlSGsGE8ne/7G0AgIELtp6VnSwr989RjJGJFbdl5S21M17YUyNt3VUyG00D25laV7fS0H/gVXXLKI16z9neXCAa2i8bp9E6vN1ZL7V0bx2of/8L91ZRF/fFNlEni8IvDHWH+vSJqLA1W88OKP9vh4ikz/7XQr/Bwd3/Lyub98w30Z8/QxA2OMz6hw+b/oaDjds7OA1WofL7bSe4rB9OkLjTfv/97pO7dL89X9AKEOO0LLLL2PKcBjlmm2vgZQmAjVzId5bu9k4+D26gFPIT1ao9Z9M9e2O6rMuyapr2dEkoqRXBLOmOqTu6YJqSUynlN3K4L2/nwkriviimoUQX3NX01VPKHZwzFJdkhEdpMLFwOXWotiHhcUlGOObfxXK1ZNZldo4pKcVNLHCleWumWW832NOvBDG08vh/WrvGJq1XJicOcHUwL3E1NXmamlvR1MzSFB2CUego7IFb8gws1DU5unts7yj/4I2egHHzVO3WYSudrBrRmIxVhpaZtsrI5KikpzgWnSgZtUl6rDpNzzxoTCvgpXKUwfwNq2WT8R/XIUlnEjlS9PBy1a51ezitgXdvtzLD3tqFJ2ZQkKes02YqgXebzXHK7B02fGSOhunj4IPnQATvWjM5ksYDE/Zqc0xMH3QHxMyjc1Ys77YprgPsNVPyzNwvDZ4HNyuUwlFAlxZGQ0CIVUSEwToiQRwlMBgtvGVKzqKPFbjkxGBnyYPVpkTzRjseEFt0v44d8CyZGfNJ8mDlKdE85cG+pFW2PZvVnqAz6lP0zHGfVrD+uLfL83atRIdIfyJXqkzfLMBwNjwPMBp5wxXckRa3MpTDDN3gOWPsBvK491xxQmrylQU89o5rSMm+3BrK5SktzTrl9rpqIy6jmKFlqmak85TPuhXPClNGOOoZPlOBBDVr7krqWLcS4X2j9SoRXWpKVCJ1mymjEgOyJAMsNpxmZRDyCb8P9QF1w4gPtHA/x+HYTWsepgpr2vEWHKdmhUfh3GoNKs+6kYNKZd+GHlwrTjPo1QIrfW+qv/7cgnHF12Fn15hfAc5FYKn/InB443XjeTssuoKEBeIihL+ZERCQuPULeOi5BD+T7lr4cY7EXB5mryZ8qjBXKHYbwqqQ2S8UxK5LYYFgrkqKJYANS0RTSljfAegIilDPInhIrMVJPdFSENYwcpcDQuA+DEoC6e9HJgtKyLo2f+C1fTtH9oxafUkh7OdJ0oXCZJv0u2lUJgukkkXY71KICl66UyQmg90UzaImqoqoK4ag0Bt6/XfulMWhA1Wvae+btZuNIiYzZ6RGkSg6nRvFEFNx+Uazo1ixnFFnxjZrtOBVciBbjRKVtnXNniMxBoCW1f+J6JJ+FODQCwowX5Bv/oIccBZ8YlFQmS9aFGRRYbAoKmpRNhVNlTFFCS9aG4ZZFAVJl4JcJj4W27QjSBVrpTdBWkEokjzkgoignHWaLta3/xYcXv00TK0gioZSwKGeBHHfKhJoslKbSszuT7rERkds8j3uWkfQlBw7qVGzwfHCFx/VwPFm62zgmmJw2hoyMlMZ0yq4X9Wq1d0W29CQwn5NK6RkuWOxtEEe/8/l5zfsVisR5AWEWEVEGKwjErTONoMVYqPfp424ya0Ha4N4YaXzUuXIRVOLbMDzM+XYAsw+ZpKcVYuUGt7giK6ycqvBsJkhwxKzkJRQMGhnpE/3z7hH0qfr59gtaR12GyQblw+wiMsAjKFlu9CMfM4KbqzvN0+Gi+SLyZZLiY3XcgPNQj7BJ/RXZn+7OV3fU58cyUcLmSopRd1QdVkTDBkzETEBVwI3yxBVQ9IUURPIBx77vtTItD3M92ESlNY/20ifYNBVSURzDkhD5rxF/BBStWrds9xa2VnE7+sdSpt4oB1jixzLyL73sKQ1r1g4h2qCYDlzdjK7GxyGIBkZK+EsgseJ+kldwYkvA4qmj0BX0xywGmZ6JdLOoy9S/c/jHP0wBsAibCm8IUiKVjw4OrlvbGIvUISCUCAeibtoQyhkv0KPB2iCWbNeg3qVD9I8ENAjFiW5iJ9GKROj1CuVFg4c8OQUo2joqmJIpmTKAvS/JJtFWZWAoCoqSQxDIxOvZVgo6qYuQU1UgfxHDjzzM6Hqo3EiplzRDRMBX4GqBGENHktwsvLDZLd7n/gfve2f/7j3/HnupdgwmAYzXW/Xym5thnwtATCowP/wR9UEvNcaazVGjEyVEjfpnwlwWWXg3aEyUYR/zbS2aTVbmxxr633/J//Mk6QCgzYG26SnxW/QhEqsAYy3tnQTDXnj2k/Pznav3vOXv/KfLfmX+I9i5V3AgpNkTz1TefiuZ+enUs3WVOKpiQLmDQoyCYmLEuaUcRv5IiqKQZ5+roAfqspaYKkYhnwKyVFKKoQjQmGqBu6KJjKlpqjvuDmiADloioLoirKmo5JjQVDlIvQJFnHOQ6994Jm6lBGotnYMhRRRIA6TIhRTjcZ1OFofN0saDD8dPgTfHTmUOF1US4oBfzmWgEQhiedSFMLMqWJMU2DAkJwhkyOL5GoI0EgYoM0NaQa9FUNX6dZiUg1qRKUEzgBQJWISGgcSgg2vZEswrv48T4k0/kTqSAVay9qSbJMKaQqDFkpiAYla4V0fId8iF9M5BPdyfgB2RpVTzSGUoPM0nCKHZqVh8KvYwzR2Q9WxIo7Q4ExH05kmPlDv1yMHyu2wpUOj/KDAsf+ComWISPaWAAA'
s=''.join(PAYLOAD.split()); s += '=' * (-len(s) % 4)
(ROOT/'workflow.json').write_bytes(gzip.decompress(base64.b64decode(s,validate=True)))
print('工作流已释放；素材将在 ComfyUI 内上传。')

In [ ]:
# Cell 2：同步最新 ComfyUI 与节点
def sync(url,path,branch=None):
 path=Path(path)
 if (path/'.git').exists():
  sh(f"git remote set-url origin '{url}'",path); sh(f"git fetch --depth 1 origin {branch or 'HEAD'}",path); sh('git reset --hard FETCH_HEAD && git clean -fd',path)
 else:
  shutil.rmtree(path,ignore_errors=True); sh(f"git clone --depth 1 {'--branch '+branch if branch else ''} '{url}' '{path}'")
 print(path.name,sh("git log -1 --format='%h %ad %s' --date=short",path).stdout.strip()); return path
def requirements(path):
 f=Path(path)/'requirements.txt'
 if not f.exists(): return
 lines=[x for x in f.read_text(errors='ignore').splitlines() if x.strip() and not re.match(r'^(torch|torchvision|torchaudio|triton)([<=>~!\[]|$)',x.strip(),re.I)]
 t=ROOT/'requirements.safe.txt'; t.write_text('\n'.join(lines)); sh(f"{sys.executable} -m pip install -q -r '{t}'")
sync('https://github.com/Comfy-Org/ComfyUI.git',COMFY,'master'); requirements(COMFY)
sh(f"{sys.executable} -m pip install -q 'huggingface_hub[hf_xet]' requests")
cn=COMFY/'custom_nodes'; cn.mkdir(parents=True,exist_ok=True)
repos=[('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director'),('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes'),('https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git','ComfyUI-MiniMax-H3-Turbo'),('https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Manager')]
for u,n in repos: requirements(sync(u,cn/n))
print('最新 ComfyUI/Director/自定义节点已就绪。')

In [ ]:
# Cell 3：下载 A100 模型
from huggingface_hub import hf_hub_download
def model(repo,file,sub):
 src=Path(hf_hub_download(repo_id=repo,filename=file,cache_dir=str(HF),token=os.environ.get('HF_TOKEN'))).resolve(); d=COMFY/'models'/sub; d.mkdir(parents=True,exist_ok=True); dst=d/Path(file).name
 if dst.exists() or dst.is_symlink(): dst.unlink()
 try: os.link(src,dst)
 except OSError: os.symlink(src,dst)
 print('✓',dst.name)
r='Comfy-Org/MiniMax-H3'
for f,d in [('diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors','diffusion_models'),('text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors','text_encoders'),('vae/minimax_h3_video_vae_fp16.safetensors','vae'),('vae/minimax_h3_audio_vae_fp32.safetensors','vae')]: model(r,f,d)
model('larryvrh/MiniMax-H3-Turbo-Lora','minimax_h3_turbo_v4_step600_ema.safetensors','loras')
u=COMFY/'models'/'upscale_models'; u.mkdir(parents=True,exist_ok=True); p=u/'4x-UltraSharp.pth'
if not p.exists(): urllib.request.urlretrieve('https://huggingface.co/embed/upscale/resolve/main/4x-UltraSharp.pth',p)
wdir=COMFY/'user'/'default'/'workflows'; wdir.mkdir(parents=True,exist_ok=True); shutil.copy2(ROOT/'workflow.json',wdir/f'{WORKFLOW}.json')
print('模型与工作流已就绪。')

In [ ]:
# Cell 4：启动 ComfyUI；素材直接在导演台时间线上传
sh("pkill -f '/content/ComfyUI/main.py' || true",check=False)
log=open(ROOT/'comfy.log','w'); proc=subprocess.Popen([sys.executable,'main.py','--listen','0.0.0.0','--port',str(PORT),'--preview-method','auto','--disable-auto-launch'],cwd=COMFY,stdout=log,stderr=subprocess.STDOUT)
for _ in range(180):
 try:
  if urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats',timeout=2).status==200: break
 except Exception: time.sleep(2)
else: raise RuntimeError('ComfyUI 启动失败，请查看 /content/comfy.log')
print('READY：Workflows →',WORKFLOW); print('在 MiniMaxH3Director 时间线中直接上传首尾帧/参考素材。')
from google.colab import output
output.serve_kernel_port_as_window(PORT)